In [2]:
from statsbombpy import sb
import pandas as pd
import numpy as np


import warnings

pd.set_option("display.max_columns", None)

# turn off warnings for cleaner output
warnings.filterwarnings("ignore")

# retrieve 2003/2004 Premier League matches
matches = sb.matches(competition_id=2, season_id=44)

# get all the matches that Arsenal played
arsenal_matches = matches[
    (matches["home_team"] == "Arsenal") | (matches["away_team"] == "Arsenal")
]
# Get the first match Arsenal played in the dataset
match_id = arsenal_matches.iloc[0]["match_id"]
# match name for printing
match_name = (
    f"{arsenal_matches.iloc[0]['home_team']} vs {arsenal_matches.iloc[0]['away_team']}"
)
print(f"Fetching tick data for: {match_name}\n")
# get the data for every single event that occurred in the match: pass, tackle etc.
events = sb.events(match_id=match_id)
# break the [x, y] coordinates pair column into separate x and y columns
events["x"] = events["location"].apply(lambda x: x[0] if isinstance(x, list) else None)
events["y"] = events["location"].apply(lambda x: x[1] if isinstance(x, list) else None)

# add a column for how many seconds have passed in each period (details about period in dataset_info.md)
events["period_seconds"] = pd.to_timedelta(events["timestamp"]).dt.total_seconds()

# sort by period first, then by seconds passed in that period to ensure chronological order of events
# without reset_index, the row indices would be messed up because it would keep the unsorted values
# without drop=True, the old row indices would be kept as a new column. we don't need them
events = events.sort_values(by=["period", "period_seconds"]).reset_index(drop=True)

# events["delta_t"] = events.groupby("period")["period_seconds"].diff()

# events["delta_t"] = events["delta_t"].fillna(0)
# drop any events where the match hadn't started yet
game_states = events.dropna(subset=["x", "y"]).copy()

# sort again by period and seconds in period
game_states = game_states.sort_values(by=["period", "period_seconds"]).reset_index(
    drop=True
)
# group by period, then find the time spent in each event
game_states["delta_t"] = game_states.groupby("period")["period_seconds"].diff()
# the first event will always have a NaN value becuase there was no previous event to compare the time to
game_states["delta_t"] = game_states["delta_t"].fillna(0)

print(game_states[["type", "period_seconds", "delta_t", "x", "y"]].head())

Fetching tick data for: Wolverhampton Wanderers vs Arsenal

            type  period_seconds  delta_t     x     y
0           Pass           0.675    0.000  61.0  40.1
1  Ball Receipt*           1.333    0.658  61.4  43.6
2           Pass           1.390    0.057  61.9  44.0
3  Ball Receipt*           2.494    1.104  46.9  40.8
4          Carry           2.494    0.000  46.9  40.8


In [3]:
print(game_states.head())

  bad_behaviour_card ball_receipt_outcome ball_recovery_offensive  \
0                NaN                  NaN                     NaN   
1                NaN                  NaN                     NaN   
2                NaN                  NaN                     NaN   
3                NaN                  NaN                     NaN   
4                NaN                  NaN                     NaN   

  ball_recovery_recovery_failure block_offensive carry_end_location  \
0                            NaN             NaN                NaN   
1                            NaN             NaN                NaN   
2                            NaN             NaN                NaN   
3                            NaN             NaN                NaN   
4                            NaN             NaN       [46.9, 39.1]   

  clearance_aerial_won clearance_body_part clearance_head clearance_left_foot  \
0                  NaN                 NaN            NaN                 NaN

In [4]:
events_35th_min = events[events["minute"] == 35].copy()

shots_df = events[events["type"] == "Shot"].copy()
print(shots_df[shots_df["shot_outcome"] == "Goal"])

loc_df = events.loc[:, "location"]

     bad_behaviour_card ball_receipt_outcome ball_recovery_offensive  \
359                 NaN                  NaN                     NaN   
868                 NaN                  NaN                     NaN   
2091                NaN                  NaN                     NaN   
2218                NaN                  NaN                     NaN   

     ball_recovery_recovery_failure block_offensive carry_end_location  \
359                             NaN             NaN                NaN   
868                             NaN             NaN                NaN   
2091                            NaN             NaN                NaN   
2218                            NaN             NaN                NaN   

     clearance_aerial_won clearance_body_part clearance_head  \
359                   NaN                 NaN            NaN   
868                   NaN                 NaN            NaN   
2091                  NaN                 NaN            NaN   
2218        

In [5]:
import numpy as np

x_edges = np.linspace(0, 120, 7)
y_edges = np.linspace(0, 80, 5)

game_states["x_bin"] = pd.cut(
    game_states["x"], bins=x_edges, labels=False, include_lowest=True
)
game_states["y_bin"] = pd.cut(
    game_states["y"], bins=y_edges, labels=False, include_lowest=True
)

game_states["zone_id"] = (game_states["x_bin"] * 4) + game_states["y_bin"]

print(game_states[["type", "x", "y", "x_bin", "y_bin", "zone_id"]].head(10))

            type     x     y  x_bin  y_bin  zone_id
0           Pass  61.0  40.1      3      2       14
1  Ball Receipt*  61.4  43.6      3      2       14
2           Pass  61.9  44.0      3      2       14
3  Ball Receipt*  46.9  40.8      2      2       10
4          Carry  46.9  40.8      2      2       10
5           Pass  46.9  39.1      2      1        9
6  Ball Receipt*  41.0   6.2      2      0        8
7          Carry  41.0   6.2      2      0        8
8           Pass  37.6   2.9      1      0        4
9  Ball Receipt*  23.4  24.7      1      1        5


In [6]:
target_team = "Arsenal"

game_states["P"] = (game_states["possession_team"] == target_team).astype(int)

print(
    game_states[
        ["type", "minute", "second", "team", "possession_team", "P", "zone_id"]
    ].head(10)
)

            type  minute  second     team possession_team  P  zone_id
0           Pass       0       0  Arsenal         Arsenal  1       14
1  Ball Receipt*       0       1  Arsenal         Arsenal  1       14
2           Pass       0       1  Arsenal         Arsenal  1       14
3  Ball Receipt*       0       2  Arsenal         Arsenal  1       10
4          Carry       0       2  Arsenal         Arsenal  1       10
5           Pass       0       3  Arsenal         Arsenal  1        9
6  Ball Receipt*       0       4  Arsenal         Arsenal  1        8
7          Carry       0       4  Arsenal         Arsenal  1        8
8           Pass       0       8  Arsenal         Arsenal  1        4
9  Ball Receipt*       0      10  Arsenal         Arsenal  1        5


In [7]:
is_goal = (game_states["type"] == "Shot") & (game_states["shot_outcome"] == "Goal")

game_states["arsenal_goal_event"] = np.where(
    is_goal & (game_states["team"] == target_team), 1, 0
)
game_states["opponent_goal_event"] = np.where(
    is_goal & (game_states["team"] != target_team), 1, 0
)

game_states["arsenal_score"] = game_states["arsenal_goal_event"].cumsum()
game_states["opponent_score"] = game_states["opponent_goal_event"].cumsum()

game_states["delta_G"] = game_states["arsenal_score"] - game_states["opponent_score"]
game_states["delta_G"] = game_states["delta_G"].clip(-2, 2)

In [8]:
game_states["state_id"] = (
    "G:"
    + game_states["delta_G"].astype(str)
    + "_P:"
    + game_states["P"].astype(str)
    + "_Z:"
    + game_states["zone_id"].astype(str)
)

print(
    game_states[
        ["minute", "second", "team", "type", "delta_G", "P", "zone_id", "state_id"]
    ].head(15)
)

    minute  second                     team           type  delta_G  P  \
0        0       0                  Arsenal           Pass        0  1   
1        0       1                  Arsenal  Ball Receipt*        0  1   
2        0       1                  Arsenal           Pass        0  1   
3        0       2                  Arsenal  Ball Receipt*        0  1   
4        0       2                  Arsenal          Carry        0  1   
5        0       3                  Arsenal           Pass        0  1   
6        0       4                  Arsenal  Ball Receipt*        0  1   
7        0       4                  Arsenal          Carry        0  1   
8        0       8                  Arsenal           Pass        0  1   
9        0      10                  Arsenal  Ball Receipt*        0  1   
10       0      10                  Arsenal          Carry        0  1   
11       0      10  Wolverhampton Wanderers       Pressure        0  1   
12       0      12                  Ar